In [1]:
lapply(c('viridis', 'ggthemes', 'skimr'),
       function(pkg_name) { if(! pkg_name %in% installed.packages()) { install.packages(pkg_name)} } )

library(viridis)    # A nice color scheme for plots.
library(ggthemes)   # Common themes to change the look and feel of plots.
library(scales)     # Graphical scales map data to aesthetics in plots.
library(skimr)      # Better summaries of data.
library(lubridate)  # Date library from the tidyverse.
library(tidyverse)  # Data wrangling packages.
library(bigrquery)  # Data extraction from Google BigQuery

## Plot setup.
theme_set(theme_bw(base_size = 14)) # Default theme for plots.

#' Returns a data frame with a y position and a label, for use annotating ggplot boxplots.
#'
#' @param d A data frame.
#' @return A data frame with column y as max and column label as length.
get_boxplot_fun_data <- function(df) {
  return(data.frame(y = max(df), label = stringr::str_c('N = ', length(df))))
}

system("jupyter nbconvert --to notebook --execute '/home/jupyter/workspace/rw-migration-aou-rw-67cadd48/Setting_Env_Variables.ipynb' --stdout")

[[1]]
NULL

[[2]]
NULL

[[3]]
NULL

Loading required package: viridisLite


Attaching package: ‘scales’


The following object is masked from ‘package:viridis’:

    viridis_pal



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr   1.2.1     ✔ readr   2.2.0
✔ forcats 1.0.1     ✔ stringr 1.6.0
✔ ggplot2 4.0.2     ✔ tibble  3.3.1
✔ purrr   1.2.1     ✔ tidyr   1.3.2
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ readr::col_factor() masks scales::col_factor()
✖ purrr::discard()    masks scales::discard()
✖ dplyr::filter()     masks stats::filter()
✖ dplyr::lag()        masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


# Load and Merge Data

In [2]:
current_directory <- "/home/jupyter/workspace/rw-migration-aou-rw-67cadd48"
output_dir <- "/home/jupyter/workspace/rw-migration-aou-rw-67cadd48/data"

demo_file = paste0(current_directory, "/data/Demographic_and_ancestry_covariates.csv")
demo  <- read_csv(demo_file) |> 
    select(-c(income)) |> 
    rename(age = age_at_last_event)

ses_file = paste0(current_directory, "/data/SES_data.csv")
ses_data  <- read_csv(ses_file) |> 
    select(-c(education))

Rows: 135557 Columns: 51
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (9): SexGender, ancestry_pred, education, where_born, military, health...
dbl  (30): person_id, race_unknown, age_today, PC_1, PC_2, PC_3, PC_4, PC_5,...
lgl   (8): AIAN, Asian, Black, Mid, Multiple, PI, White, His
dttm  (1): basics_survey_datetime
date  (3): date_of_birth, min_date, max_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 214915 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (13): person_id, income, education, per_poverty_threshold, individual_po...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
# Select the required columns from pov_data and merge with all_data
all_data <- merge(
  demo,
  ses_data,
  by = "person_id"
)

# Format Data

In [4]:
# Convert indicator columns into a single 'Race' column
all_data2 <- all_data %>%
  pivot_longer(
    cols = c(AIAN, Asian, Black, Mid, Multiple, PI, White, race_unknown),
    names_to = "racial_identity",
    values_to = "indicator"
  ) %>%
  filter(indicator == 1) %>%
  select(-indicator)


In [5]:
all_data2$education<-factor(all_data2$education,
                             levels = c("Less than high school degree or equivalent",
                                        "Twelve or GED",
                                         "College One to Three",
                                       "College graduate",
                                        "Advanced degree"),
                             ordered = TRUE)

all_data2$income_grouped<-factor(all_data2$income_grouped,
                             levels = c("<50k",
                                       "50k-100k",
                                       "100k-150k",
                                       ">150k"),
                             ordered = TRUE)


all_data2$racial_identity<-factor(all_data2$racial_identity,
                             levels = c("AIAN",
                                       "Asian",
                                       "Black",
                                       "Mid",
                                       "Multiple",
                                       "PI",
                                       "White",
                                       "race_unknown"),
                             ordered = TRUE)
# Rename some levels
levels(all_data2$racial_identity)[levels(all_data2$racial_identity) == "AIAN"] <- "American Indian or Alaska Native"
levels(all_data2$racial_identity)[levels(all_data2$racial_identity) == "Mid"] <- "Middle Eastern or North African"
levels(all_data2$racial_identity)[levels(all_data2$racial_identity) == "PI"] <- "Native Hawaiian or Other Pacific Islander"
levels(all_data2$racial_identity)[levels(all_data2$racial_identity) == "race_unknown"] <- "PNA or Skip"
names(all_data2)[names(all_data2) == "racial_identity"] <- "racial identity"



all_data2$disabled<-factor(all_data2$disabled,
                             levels = c("Yes",
                                       "No",
                                       "Missing"),
                             ordered = TRUE)
names(all_data2)[names(all_data2) == "disabled"] <- "disability"
levels(all_data2$disability)[levels(all_data2$disability) == "Missing"] <- "PNA or Skip"




names(all_data2)[names(all_data2) == "His"] <- "Hispanic"
all_data2$Hispanic<-factor(all_data2$Hispanic)
levels(all_data2$Hispanic)[levels(all_data2$Hispanic) == TRUE] <- "Yes"
levels(all_data2$Hispanic)[levels(all_data2$Hispanic) == FALSE] <- "No"
all_data2$Hispanic<-factor(all_data2$Hispanic,
                             levels = c("Yes",
                                       "No"),
                             ordered = TRUE)



all_data2$sexual_orientation<-factor(all_data2$sexual_orientation,
                             levels = c("Queer",
                                       "Straight",
                                       "PNA or Skip"),
                             ordered = TRUE)
names(all_data2)[names(all_data2) == "sexual_orientation"] <- "sexual orientation"


all_data2$SexGender<-factor(all_data2$SexGender)
levels(all_data2$SexGender)[levels(all_data2$SexGender) == "Cis_female"] <- "Cisgender Female"
levels(all_data2$SexGender)[levels(all_data2$SexGender) == "Cis_male"] <- "Cisgender Male"
levels(all_data2$SexGender)[levels(all_data2$SexGender) == "SGM"] <- "Sexual and Gender Minority (SGM)"
names(all_data2)[names(all_data2) == "SexGender"] <- "sex/gender"


names(all_data2)[names(all_data2) == "age_at_last_event"] <- "age"


all_data2$where_born<-factor(all_data2$where_born)
levels(all_data2$where_born)[levels(all_data2$where_born) == "USA"] <- "Yes"
levels(all_data2$where_born)[levels(all_data2$where_born) == "elsewhere"] <- "No"
all_data2$where_born<-factor(all_data2$where_born,
                             levels = c("Yes",
                                       "No",
                                       "Skip"),
                             ordered = TRUE)
names(all_data2)[names(all_data2) == "where_born"] <- "born in the USA?"

all_data2$military[is.na(all_data2$military)] <- "Skip or PNA"
all_data2$military<-factor(all_data2$military,
                             levels = c("Yes",
                                       "No",
                                       "Skip or PNA"),
                             ordered = TRUE)
names(all_data2)[names(all_data2) == "military"] <- "military active duty (past or present)"


all_data2$healthcare <- as.character(all_data2$healthcare)
all_data2$healthcare[all_data2$healthcare == "Skip"] <- "Skip or NA"
all_data2$healthcare[is.na(all_data2$healthcare)] <- "Skip or NA"


all_data2$healthcare<-factor(all_data2$healthcare,
                             levels = c("Employer Or Union",
                                       "Medicaid",
                                       "Medicare",
                                       "Other",
                                       "Purchased",
                                       "VA or Military",
                                        "None", 
                                       "Skip or NA"),
                             ordered = TRUE)
names(all_data2)[names(all_data2) == "healthcare"] <- "health insurance"


# Get full cohort summary

In [7]:
# Columns to summarize
cols <- c(
  "age", "sex/gender", "ancestry_pred", "racial identity", "Hispanic",
  "education", "income_grouped", 
    "per_poverty_threshold", "fraction_assisted_income", "fraction_no_high_school_education",
    "lower_median_income", "fraction_no_health_insurance", "fraction_poverty", 
    "fraction_vacant_housing", "deprivation_index", "racial identity",   
    "sexual orientation", "disability", "born in the USA?",
  "military active duty (past or present)", "health insurance"
)


selected_data <- all_data2 |> select(cols)

# Unified summarizer
summarize_column_detailed_cohorts <- function(df, cols) {
  summary_list <- list()
  message("Number of rows: ", nrow(df))

  for (col_name in cols) {
    column <- df[[col_name]]

    if (is.numeric(column)) {
      # Numeric summary → single row
      summary_list[[col_name]] <- data.frame(
        Variable = col_name,
        Value = NA,
        Summary = sprintf("%.2f (± %.2f)", 
                          mean(column, na.rm = TRUE), 
                          sd(column, na.rm = TRUE))
      )
    } else {
      # Categorical summary → multiple rows
      counts <- as.data.frame(table(column, useNA = "ifany"))
      counts$Percent <- round(100 * counts$Freq / sum(counts$Freq), 1)
      summary_list[[col_name]] <- data.frame(
        Variable = col_name,
        Value = as.character(counts$column),
        Summary = paste0(counts$Freq, " (", counts$Percent, "%)")
      )
    }
  }

  # Combine all results safely
  do.call(rbind, summary_list)
}


detailed_summary_table <- summarize_column_detailed_cohorts(selected_data, cols) 
rownames(detailed_summary_table) <- NULL


# Preview
head(detailed_summary_table, 60)


write_excel_csv(
  detailed_summary_table,
  file.path(output_dir, "Whole_sample_demographics.csv")
)


Number of rows: 104908



,Variable,Value,Summary
,<chr>,<chr>,<chr>
1,age,NA,57.30 (± 16.49)
2,sex/gender,Cisgender Female,63269 (60.3%)
3,sex/gender,Cisgender Male,39226 (37.4%)
4,sex/gender,Sexual and Gender Minority (SGM),2413 (2.3%)
5,ancestry_pred,afr,19368 (18.5%)
6,ancestry_pred,amr,10465 (10%)
7,ancestry_pred,eas,1375 (1.3%)
8,ancestry_pred,eur,72424 (69%)
9,ancestry_pred,mid,330 (0.3%)


# Get cohort summary for smaller richer SDoH survey cohort

In [14]:
# Imputed individual-level SDoH data from Social Determinants of Health Measurement Models V8 community workspace
name_of_file_in_bucket <- 'imputed_individual_level_SDOH.csv'
system(paste0("gsutil cp gs://rw-migration-aou-rw-55feedb6/notebooks/imputed_individual_level_SDOH.csv ."), intern=T)

sdoh_var <-c('Own Home', 'Health Literacy', 'Housing Quality', 'Housing Instability',
             'Food Insecurity', 'Walkability', 'Loneliness', 'Crime', 'Physical Disorder', 'Social Disorder',
             'Everyday Discrimination', 'Medical Discrimination', 'Social Cohesion', 'Stress', 
             'Social Support', 'Spiritual Experiences', 'Health Coverage', 'Healthcare Utilization',
             'Delayed Care', 'Can\'t afford care', 'Worried Pay', 'Respect'
)

sdoh_data <- read_csv(name_of_file_in_bucket) |> select(c("person_id", sdoh_var))

sdoh_survey_data <- merge(
  all_data2,
  sdoh_data,
  by = "person_id"
)


selected_data <- sdoh_survey_data |> select(cols)

detailed_summary_table <- summarize_column_detailed_cohorts(selected_data, cols) 
rownames(detailed_summary_table) <- NULL

# Preview
head(detailed_summary_table, 60)

write_excel_csv(
  detailed_summary_table,
  file.path(output_dir, "Richer_sdoh_sample_demographics.csv")
)


character(0)

Rows: 54313 Columns: 37
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (1): SexGender
dbl (30): person_id, race_unknown, record_depth, visit_frequency, age, Educa...
lgl  (6): Black, Mid, Multiple, White, His, AANHPI

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Number of rows: 47455



,Variable,Value,Summary
,<chr>,<chr>,<chr>
1,age,NA,59.51 (± 15.61)
2,sex/gender,Cisgender Female,29094 (61.3%)
3,sex/gender,Cisgender Male,17386 (36.6%)
4,sex/gender,Sexual and Gender Minority (SGM),975 (2.1%)
5,ancestry_pred,afr,3967 (8.4%)
6,ancestry_pred,amr,2816 (5.9%)
7,ancestry_pred,eas,594 (1.3%)
8,ancestry_pred,eur,39572 (83.4%)
9,ancestry_pred,mid,129 (0.3%)


# Case Control Deomographics

In [15]:
name_of_file_in_bucket <- 'Case_Control_df.csv'
system(paste0("gsutil cp gs://rw-migration-aou-rw-55feedb6/notebooks/Case_Control_df.csv ."), intern=T)
case_control  <- read_csv(name_of_file_in_bucket)

character(0)

Rows: 265954 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl (10): person_id, Asthma, Afib, BreastC, CKD, CHD, HyperC, ProstateC, t2d...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [16]:
merged_df<-merge(all_data2, case_control)
colnames(merged_df)

[1] "person_id"                             
 [2] "date_of_birth"                         
 [3] "Hispanic"                              
 [4] "sex/gender"                            
 [5] "age_today"                             
 [6] "ancestry_pred"                         
 [7] "PC_1"                                  
 [8] "PC_2"                                  
 [9] "PC_3"                                  
[10] "PC_4"                                  
[11] "PC_5"                                  
[12] "PC_6"                                  
[13] "PC_7"                                  
[14] "PC_8"                                  
[15] "PC_9"                                  
[16] "PC_10"                                 
[17] "PC_11"                                 
[18] "PC_12"                                 
[19] "PC_13"                                 
[20] "PC_14"                                 
[21] "PC_15"                                 
[22] "PC_16"                                 
[23] "number_living"                         
[24] "education"                             
[25] "born in the USA?"                      
[26] "military active duty (past or present)"
[27] "health insurance"                      
[28] "disability"                            
[29] "sexual orientation"                    
[30] "LGBTQIA"                               
[31] "basics_survey_datetime"                
[32] "income_grouped"                        
[33] "min_date"                              
[34] "max_date"                              
[35] "ehr_length"                            
[36] "relative_health"                       
[37] "record_depth"                          
[38] "visit_frequency"                       
[39] "age"                                   
[40] "record_depth_winsorized"               
[41] "visit_frequency_winsorized"            
[42] "age2"                                  
[43] "income"                                
[44] "per_poverty_threshold"                 
[45] "individual_poverty"                    
[46] "american_community_survey_year"        
[47] "fraction_assisted_income"              
[48] "fraction_no_high_school_education"     
[49] "lower_median_income"                   
[50] "fraction_no_health_insurance"          
[51] "fraction_poverty"                      
[52] "fraction_vacant_housing"               
[53] "deprivation_index"                     
[54] "racial identity"                       
[55] "Asthma"                                
[56] "Afib"                                  
[57] "BreastC"                               
[58] "CKD"                                   
[59] "CHD"                                   
[60] "HyperC"                                
[61] "ProstateC"                             
[62] "t2d"                                   
[63] "t1d"

In [17]:
diseases<-c("Asthma", "Afib", "BreastC", "CKD", "CHD", "HyperC", "ProstateC", "t2d", "t1d")


In [18]:
summarize_disease_cohort <- function(cohort, diseases, case_value = 1) {
  output_df <- data.frame()
  
  for (disease in diseases) {
    if (disease %in% colnames(cohort)) {
      df <- cohort[cohort[[disease]] == case_value & !is.na(cohort[[disease]]), ]
      n <- nrow(df)
        
      output_df[disease, "Disease"] <- disease
     
        
      #Case control
       if (case_value == 1) {
          output_df[disease, "Status"] <- "Case"   
       } else { output_df[disease, "Status"] <- "Control" }
    
      output_df[disease, "Sample size"] <- n
      
      if (n > 0) {
        # Median age (SD)
        output_df[disease, "Age (sd)"] <- paste0(
          round(median(df$age, na.rm = TRUE), 2), 
          " (", round(sd(df$age, na.rm = TRUE), 2), ")"
        )

        # Helper function for count + percent
        fmt <- function(x) {
          count <- sum(x, na.rm = TRUE)
          if (count > 0 & count < 20) return("<20")
          percent <- round(100 * count / n, 1)
          paste0(count, " (", percent, "%)")
        }

        # Sex/Gender
        output_df[disease, "Cisgender Male"] <- fmt(df$`sex/gender` == "Cisgender Male")
        output_df[disease, "Cisgender Female"] <- fmt(df$`sex/gender` == "Cisgender Female")
        output_df[disease, "SGM"] <- fmt(df$`sex/gender` == "Sexual and Gender Minority (SGM)")

        # Race
        races <- c("Asian", "Black", "Middle Eastern or North African", "Multiple", 
                   "Native Hawaiian or Other Pacific Islander", "White")
        for (race in races) {
          output_df[disease, race] <- fmt(df$`racial identity` == race)
        }

        # Ethnicity
        output_df[disease, "Hispanic"] <- fmt(df$Hispanic == "Yes")

        # Record depth (numeric only)
        output_df[disease, "Median Record Depth"] <- median(na.omit(df$record_depth))

        # Disability (if column exists)
        output_df[disease, "Disability"] <- if ("disability" %in% colnames(df)) fmt(df$disability == "Yes") else NA

        # LGBTQIA
        output_df[disease, "LGBTQIA"] <- fmt(df$LGBTQIA == 1)

        # Income
        output_df[disease, "Income <50k"] <- fmt(df$income_grouped == "<50k")
        output_df[disease, "Income >150k"] <- fmt(df$income_grouped == ">150k")

        # Education
        output_df[disease, "Advanced Degree"]            <- fmt(df$education == "Advanced degree")
        output_df[disease, "Less than HighSchool"] <- fmt(df$education == "Less than high school degree or equivalent")
      }
    }
  }
  
  return(output_df)
}


In [21]:
cases<-summarize_disease_cohort(merged_df, diseases, 1)
rownames(cases) <- NULL
controls<-summarize_disease_cohort(merged_df, diseases, 0)
rownames(controls) <- NULL


# Interleave rows
n <- nrow(cases)
interleaved <- bind_rows(
  cases %>% mutate(.order = (1:n) * 2 - 1),
  controls %>% mutate(.order = (1:n) * 2)
) %>%
  arrange(.order) %>%
  select(-.order)

# View result
interleaved <- interleaved %>%
  relocate(Status, .after = 2)

    
#Save
interleaved
write_excel_csv(
  interleaved,
  file.path(output_dir, "Case_control_demographics.csv")
)
    

Disease,Status,Sample size,Age (sd),Cisgender Male,Cisgender Female,SGM,Asian,Black,Middle Eastern or North African,⋯,Native Hawaiian or Other Pacific Islander,White,Hispanic,Median Record Depth,Disability,LGBTQIA,Income <50k,Income >150k,Advanced Degree,Less than HighSchool
<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Asthma,Case,12315,54 (16.06),3088 (25.1%),8927 (72.5%),300 (2.4%),162 (1.3%),2312 (18.8%),33 (0.3%),⋯,<20,7823 (63.5%),1688 (13.7%),143,2607 (21.2%),1628 (13.2%),5935 (48.2%),1712 (13.9%),3098 (25.2%),692 (5.6%)
Asthma,Control,18005,54 (17.25),7852 (43.6%),9733 (54.1%),420 (2.3%),566 (3.1%),3516 (19.5%),103 (0.6%),⋯,<20,10886 (60.5%),2625 (14.6%),34,2333 (13%),1935 (10.7%),7941 (44.1%),2968 (16.5%),5178 (28.8%),1179 (6.5%)
Afib,Case,8222,72 (11.92),4693 (57.1%),3301 (40.1%),228 (2.8%),61 (0.7%),811 (9.9%),24 (0.3%),⋯,<20,6644 (80.8%),429 (5.2%),189,2286 (27.8%),677 (8.2%),3410 (41.5%),1189 (14.5%),2500 (30.4%),309 (3.8%)
Afib,Control,17685,63 (15.02),6433 (36.4%),10841 (61.3%),411 (2.3%),237 (1.3%),3554 (20.1%),52 (0.3%),⋯,<20,11145 (63%),2237 (12.6%),183,3749 (21.2%),1735 (9.8%),8601 (48.6%),2342 (13.2%),4460 (25.2%),1276 (7.2%)
BreastC,Case,5324,68 (11.53),0 (0%),5220 (98%),104 (2%),80 (1.5%),631 (11.9%),<20,⋯,0 (0%),4008 (75.3%),443 (8.3%),172,988 (18.6%),337 (6.3%),1944 (36.5%),996 (18.7%),1766 (33.2%),167 (3.1%)
BreastC,Control,59963,56 (16.32),0 (0%),57671 (96.2%),2292 (3.8%),1115 (1.9%),11102 (18.5%),217 (0.4%),⋯,22 (0%),38401 (64%),7743 (12.9%),95,10553 (17.6%),7113 (11.9%),28821 (48.1%),8124 (13.5%),15427 (25.7%),3454 (5.8%)
CKD,Case,9258,72 (12.16),4365 (47.1%),4849 (52.4%),44 (0.5%),77 (0.8%),2203 (23.8%),26 (0.3%),⋯,<20,5888 (63.6%),771 (8.3%),201,2583 (27.9%),567 (6.1%),5081 (54.9%),815 (8.8%),2091 (22.6%),566 (6.1%)
CKD,Control,31058,48 (15.25),10460 (33.7%),20598 (66.3%),0 (0%),944 (3%),4071 (13.1%),187 (0.6%),⋯,<20,20548 (66.2%),4817 (15.5%),83,4775 (15.4%),3165 (10.2%),13277 (42.7%),5353 (17.2%),8507 (27.4%),1649 (5.3%)
CHD,Case,14863,71 (11.15),8783 (59.1%),5719 (38.5%),361 (2.4%),157 (1.1%),2266 (15.2%),47 (0.3%),⋯,<20,10629 (71.5%),1258 (8.5%),200,4106 (27.6%),1256 (8.5%),7260 (48.8%),1861 (12.5%),3738 (25.1%),950 (6.4%)


# Case Control Demographics Richer SDoH cohort

In [20]:
sdoh_survey_data_case_control<-merge(sdoh_survey_data, case_control)
diseases_short<-c("Asthma", "Afib",  "CKD", "CHD", "HyperC", "t2d")

In [22]:
cases<-summarize_disease_cohort(sdoh_survey_data_case_control, diseases, 1)
rownames(cases) <- NULL
controls<-summarize_disease_cohort(sdoh_survey_data_case_control, diseases, 0)
rownames(controls) <- NULL


# Interleave rows
n <- nrow(cases)
interleaved <- bind_rows(
  cases %>% mutate(.order = (1:n) * 2 - 1),
  controls %>% mutate(.order = (1:n) * 2)
) %>%
  arrange(.order) %>%
  select(-.order)

# View result
interleaved <- interleaved %>%
  relocate(Status, .after = 2)

    
#Save
interleaved
write_excel_csv(
  interleaved,
  file.path(output_dir, "Case_control_demographics_richer_sdoh.csv")
)


Disease,Status,Sample size,Age (sd),Cisgender Male,Cisgender Female,SGM,Asian,Black,Middle Eastern or North African,⋯,Native Hawaiian or Other Pacific Islander,White,Hispanic,Median Record Depth,Disability,LGBTQIA,Income <50k,Income >150k,Advanced Degree,Less than HighSchool
<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Asthma,Case,5557,58 (15.44),1384 (24.9%),4043 (72.8%),130 (2.3%),70 (1.3%),532 (9.6%),<20,⋯,<20,4369 (78.6%),432 (7.8%),158.0,1551 (27.9%),724 (13%),1836 (33%),1061 (19.1%),1884 (33.9%),76 (1.4%)
Asthma,Control,7735,59 (16.65),3329 (43%),4251 (55%),155 (2%),212 (2.7%),592 (7.7%),31 (0.4%),⋯,<20,6100 (78.9%),692 (8.9%),39.0,1257 (16.3%),835 (10.8%),1948 (25.2%),1849 (23.9%),3074 (39.7%),156 (2%)
Afib,Case,4043,72 (11.22),2283 (56.5%),1661 (41.1%),99 (2.4%),27 (0.7%),209 (5.2%),<20,⋯,<20,3577 (88.5%),109 (2.7%),176.0,1319 (32.6%),326 (8.1%),1209 (29.9%),720 (17.8%),1501 (37.1%),39 (1%)
Afib,Control,7623,66 (14.05),2706 (35.5%),4763 (62.5%),154 (2%),105 (1.4%),675 (8.9%),<20,⋯,0 (0%),6202 (81.4%),426 (5.6%),203.0,2050 (26.9%),748 (9.8%),2276 (29.9%),1438 (18.9%),2712 (35.6%),111 (1.5%)
BreastC,Case,2806,68 (10.75),0 (0%),2759 (98.3%),47 (1.7%),34 (1.2%),198 (7.1%),<20,⋯,0 (0%),2357 (84%),129 (4.6%),169.0,586 (20.9%),176 (6.3%),783 (27.9%),609 (21.7%),1107 (39.5%),29 (1%)
BreastC,Control,27215,58 (15.7),0 (0%),26296 (96.6%),919 (3.4%),470 (1.7%),2528 (9.3%),101 (0.4%),⋯,<20,21498 (79%),2012 (7.4%),101.0,6060 (22.3%),3208 (11.8%),8887 (32.7%),5132 (18.9%),9254 (34%),498 (1.8%)
CKD,Case,3799,72 (10.53),1829 (48.1%),1962 (51.6%),<20,33 (0.9%),541 (14.2%),<20,⋯,<20,2945 (77.5%),150 (3.9%),210.0,1394 (36.7%),218 (5.7%),1520 (40%),465 (12.2%),1174 (30.9%),78 (2.1%)
CKD,Control,13810,52 (15.12),4565 (33.1%),9245 (66.9%),0 (0%),366 (2.7%),800 (5.8%),78 (0.6%),⋯,<20,11032 (79.9%),1323 (9.6%),94.0,2741 (19.8%),1495 (10.8%),3843 (27.8%),3202 (23.2%),4891 (35.4%),256 (1.9%)
CHD,Case,6274,72 (9.63),3831 (61.1%),2317 (36.9%),126 (2%),46 (0.7%),408 (6.5%),<20,⋯,<20,5344 (85.2%),253 (4%),200.0,2165 (34.5%),508 (8.1%),2055 (32.8%),1079 (17.2%),2144 (34.2%),95 (1.5%)


# SDoH Quintile Deomographics

In [23]:
sdoh_var <- c(
  "per_poverty_threshold", "education",
  "fraction_assisted_income", "fraction_no_high_school_education",
  "lower_median_income", "fraction_no_health_insurance",
  "fraction_poverty", "fraction_vacant_housing",
  "deprivation_index"
)

merged_df <- merged_df %>%
  mutate(across(all_of(sdoh_var), ~ as.numeric(as.character(.))))

#Compute Q1 and Q5
quantiles <- merged_df %>%
  pivot_longer(
    cols = all_of(sdoh_var),
    names_to = "domain",
    values_to = "value"
  ) %>%
  group_by(domain) %>%
  summarize(
    q1 = quantile(value, 0.2, na.rm = TRUE),
    q5 = quantile(value, 0.8, na.rm = TRUE),
    .groups = "drop"
  )

#Add Q1 / Q5 indicator columns back to original data
for (v in sdoh_var) {
  q1_val <- quantiles$q1[quantiles$domain == v]
  q5_val <- quantiles$q5[quantiles$domain == v]

  merged_df[[paste0(v, "_Q1")]] <- merged_df[[v]] <= q1_val
  merged_df[[paste0(v, "_Q5")]] <- merged_df[[v]] >= q5_val
}



Warning message:
“There was 1 warning in `mutate()`.
ℹ In argument: `across(all_of(sdoh_var), ~as.numeric(as.character(.)))`.
Caused by warning:
! NAs introduced by coercion”


In [24]:
summarize_disease_sdoh_quintiles <- function(
  cohort,
  diseases,
  sdoh_vars,
  quintiles = c("Q1", "Q5")
) {

  output_df <- data.frame(stringsAsFactors = FALSE)

  for (sdoh in sdoh_vars) {
    for (q in quintiles) {

      # works even if sdoh has spaces/symbols
      q_col <- paste0(sdoh, "_", q)

      if (!(q_col %in% names(cohort))) next

      stratum_df <- cohort[
        !is.na(cohort[[q_col]]) & cohort[[q_col]] == TRUE,
      , drop = FALSE]

      for (disease in diseases) {

        if (!(disease %in% names(cohort))) next

        df <- stratum_df[
          !is.na(stratum_df[[disease]]) & stratum_df[[disease]] == 1,
        , drop = FALSE]

        n <- nrow(df)

        row_id <- paste(sdoh, q, disease, sep = "_")

        output_df[row_id, "SDoH"]        <- sdoh
        output_df[row_id, "Stratum"]     <- q
        output_df[row_id, "Disease"]     <- disease
        output_df[row_id, "Sample size"] <- n

        if (n == 0) next

        # ---------------- helper ----------------
        fmt <- function(x) {
          count <- sum(x, na.rm = TRUE)
          if (count > 0 && count < 20) return("<20")
          pct <- round(100 * count / n, 1)
          paste0(count, " (", pct, "%)")
        }

        # ---------------- age ----------------
        output_df[row_id, "Age (median, sd)"] <- paste0(
          round(median(df$age, na.rm = TRUE), 2),
          " (", round(sd(df$age, na.rm = TRUE), 2), ")"
        )

        # ---------------- sex/gender ----------------
        output_df[row_id, "Cisgender Male"]   <- fmt(df[["sex/gender"]] == "Cisgender Male")
        output_df[row_id, "Cisgender Female"] <- fmt(df[["sex/gender"]] == "Cisgender Female")
        output_df[row_id, "SGM"]              <- fmt(df[["sex/gender"]] == "Sexual and Gender Minority (SGM)")

        # ---------------- race ----------------
        races <- c(
          "Asian",
          "Black",
          "Middle Eastern or North African",
          "Multiple",
          "Native Hawaiian or Other Pacific Islander",
          "White"
        )

        for (race in races) {
          output_df[row_id, race] <- fmt(df[["racial identity"]] == race)
        }

        # ---------------- ethnicity ----------------
        output_df[row_id, "Hispanic"] <- fmt(df$Hispanic == "Yes")

        # ---------------- record depth ----------------
        output_df[row_id, "Median Record Depth"] <-
          round(median(df$record_depth, na.rm = TRUE), 2)

        # ---------------- disability ----------------
        output_df[row_id, "Disability"] <-
          if ("disability" %in% names(df)) fmt(df$disability == "Yes") else NA

        # ---------------- LGBTQIA ----------------
        output_df[row_id, "LGBTQIA"] <-
          if ("LGBTQIA" %in% names(df)) fmt(df$LGBTQIA == 1) else NA

        # ---------------- income ----------------
        output_df[row_id, "Income <50k"]  <- fmt(df$income_grouped == "<50k")
        output_df[row_id, "Income >150k"] <- fmt(df$income_grouped == ">150k")

        # ---------------- education ----------------
        output_df[row_id, "Advanced Degree"] <-
          fmt(df$education == 1)

        output_df[row_id, "Less than HighSchool"] <-
          fmt(df$education == 5)
      }
    }
  }

  rownames(output_df) <- NULL
  output_df
}

table1 <- summarize_disease_sdoh_quintiles(
  cohort   = merged_df,
  diseases = diseases,
  sdoh_vars = sdoh_var
)

table1 
    
#Save
write_excel_csv(
  table1,
  file.path(output_dir, "Case_SDoH_quintiles_demographics.csv")
)

SDoH,Stratum,Disease,Sample size,"Age (median, sd)",Cisgender Male,Cisgender Female,SGM,Asian,Black,⋯,Native Hawaiian or Other Pacific Islander,White,Hispanic,Median Record Depth,Disability,LGBTQIA,Income <50k,Income >150k,Advanced Degree,Less than HighSchool
<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
per_poverty_threshold,Q1,Asthma,3083,46 (14.61),597 (19.4%),2372 (76.9%),114 (3.7%),<20,1250 (40.5%),⋯,<20,966 (31.3%),751 (24.4%),114.0,834 (27.1%),550 (17.8%),3083 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,Afib,1163,65 (13.22),522 (44.9%),583 (50.1%),58 (5%),<20,353 (30.4%),⋯,0 (0%),546 (46.9%),220 (18.9%),176.0,359 (30.9%),144 (12.4%),1163 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,BreastC,665,64 (11.85),0 (0%),646 (97.1%),<20,<20,208 (31.3%),⋯,0 (0%),221 (33.2%),207 (31.1%),173.0,207 (31.1%),48 (7.2%),665 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,CKD,2080,64 (12.77),758 (36.4%),1300 (62.5%),22 (1.1%),<20,995 (47.8%),⋯,<20,578 (27.8%),436 (21%),175.0,557 (26.8%),165 (7.9%),2080 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,CHD,3083,63 (11.85),1407 (45.6%),1590 (51.6%),86 (2.8%),<20,1162 (37.7%),⋯,<20,1126 (36.5%),687 (22.3%),173.0,888 (28.8%),331 (10.7%),3083 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,HyperC,4389,62 (12.3),1436 (32.7%),2826 (64.4%),127 (2.9%),39 (0.9%),1495 (34.1%),⋯,<20,1518 (34.6%),1178 (26.8%),192.0,1162 (26.5%),454 (10.3%),4389 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,ProstateC,290,70 (8.95),274 (94.5%),0 (0%),<20,<20,114 (39.3%),⋯,0 (0%),108 (37.2%),52 (17.9%),200.5,70 (24.1%),39 (13.4%),290 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,t2d,4834,60 (12.22),1590 (32.9%),3105 (64.2%),139 (2.9%),33 (0.7%),2028 (42%),⋯,<20,1375 (28.4%),1258 (26%),127.0,1462 (30.2%),523 (10.8%),4834 (100%),0 (0%),0 (0%),0 (0%)
per_poverty_threshold,Q1,t1d,267,51 (15.01),95 (35.6%),164 (61.4%),<20,<20,91 (34.1%),⋯,<20,112 (41.9%),57 (21.3%),102.0,95 (35.6%),30 (11.2%),267 (100%),0 (0%),0 (0%),0 (0%)


# Age bucket demographics

In [25]:
merged_df <- merged_df %>%
  mutate(age_buckets = case_when(
    age <= 35 ~ "18_35",
    age <= 50 ~ "36_50",
    age <= 64 ~ "51_64",
    TRUE      ~ "65_105"
  ))

In [27]:
summarize_disease_age_buckets <- function(
  cohort,
  diseases
) {

  output_df <- data.frame()
  
  buckets <- unique(cohort$age_buckets)

  for (q in buckets) {

    stratum_df <- cohort[
      cohort$age_buckets == q & !is.na(cohort$age_buckets),
      ,
      drop = FALSE
    ]

    for (disease in diseases) {

      if (!disease %in% colnames(cohort)) next

      df <- stratum_df[
        stratum_df[[disease]] == 1 & !is.na(stratum_df[[disease]]),
        ,
        drop = FALSE
      ]

      n <- nrow(df)

      row_id <- paste(disease, q, sep = "_")
        
      output_df[row_id, "Disease"]     <- disease
      output_df[row_id, "Stratum"]     <- q
      output_df[row_id, "Sample size"] <- n

      if (n == 0) next

      # ---- Helper ----
      fmt <- function(x) {
        count <- sum(x, na.rm = TRUE)
        if (count > 0 & count < 20) return("<20")
        pct <- round(100 * count / n, 1)
        paste0(count, " (", pct, "%)")
      }

      # ---- Age ----
      output_df[row_id, "Age (median, sd)"] <- paste0(
        round(median(df$age, na.rm = TRUE), 2),
        " (", round(sd(df$age, na.rm = TRUE), 2), ")"
      )

      # ---- Sex/Gender ----
      output_df[row_id, "Cisgender Male"]   <-
        fmt(as.character(df$`sex/gender`) == "Cisgender Male")
      output_df[row_id, "Cisgender Female"] <-
        fmt(as.character(df$`sex/gender`) == "Cisgender Female")
      output_df[row_id, "SGM"]              <-
        fmt(as.character(df$`sex/gender`) == "Sexual and Gender Minority (SGM)")

      # ---- Race ----
      races <- c(
        "Asian", "Black", "Middle Eastern or North African",
        "Multiple", "Native Hawaiian or Other Pacific Islander", "White"
      )

      for (race in races) {
        output_df[row_id, race] <-
          fmt(as.character(df$`racial identity`) == race)
      }

      # ---- Ethnicity ----
      output_df[row_id, "Hispanic"] <-
        fmt(df$Hispanic == "Yes")

      # ---- Record depth ----
      output_df[row_id, "Median Record Depth"] <-
        median(df$record_depth, na.rm = TRUE)

      # ---- Disability ----
      output_df[row_id, "Disability"] <-
        if ("disability" %in% colnames(df))
          fmt(df$disability == "Yes") else NA

      # ---- LGBTQIA ----
      output_df[row_id, "LGBTQIA"] <-
        fmt(df$LGBTQIA == 1)

      # ---- Income ----
      output_df[row_id, "Income <50k"]  <-
        fmt(df$income_grouped == "<50k")
      output_df[row_id, "Income >150k"] <-
        fmt(df$income_grouped == ">150k")

      # ---- Education ----
      output_df[row_id, "Advanced Degree"] <-
        fmt(df$education == 1)
      output_df[row_id, "Less than HighSchool"] <-
        fmt(df$education == 5)
    }
  }

  output_df
}

table1 <- summarize_disease_age_buckets(
  cohort   = merged_df,
  diseases = diseases
)

save_table <- table1[order(table1$Disease),]
save_table

write_excel_csv(
  save_table,
  file.path(output_dir, "Case_age_buckets_demographics.csv")
)

,Disease,Stratum,Sample size,"Age (median, sd)",Cisgender Male,Cisgender Female,SGM,Asian,Black,Middle Eastern or North African,⋯,Native Hawaiian or Other Pacific Islander,White,Hispanic,Median Record Depth,Disability,LGBTQIA,Income <50k,Income >150k,Advanced Degree,Less than HighSchool
,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Afib_65_105,Afib,65_105,6195,76 (6.92),3674 (59.3%),2349 (37.9%),172 (2.8%),42 (0.7%),411 (6.6%),20 (0.3%),⋯,<20,5279 (85.2%),239 (3.9%),208.0,1760 (28.4%),445 (7.2%),2395 (38.7%),838 (13.5%),0 (0%),0 (0%)
Afib_51_64,Afib,51_64,1505,60 (3.82),806 (53.6%),667 (44.3%),32 (2.1%),<20,304 (20.2%),<20,⋯,0 (0%),1030 (68.4%),123 (8.2%),153.0,407 (27%),137 (9.1%),740 (49.2%),279 (18.5%),0 (0%),0 (0%)
Afib_18_35,Afib,18_35,100,31 (3.85),33 (33%),55 (55%),<20,<20,<20,0 (0%),⋯,0 (0%),68 (68%),<20,75.5,31 (31%),37 (37%),55 (55%),<20,0 (0%),0 (0%)
Afib_36_50,Afib,36_50,422,45 (4.27),180 (42.7%),230 (54.5%),<20,<20,83 (19.7%),<20,⋯,0 (0%),267 (63.3%),57 (13.5%),112.0,88 (20.9%),58 (13.7%),220 (52.1%),65 (15.4%),0 (0%),0 (0%)
Asthma_65_105,Asthma,65_105,3559,72 (5.78),1143 (32.1%),2341 (65.8%),75 (2.1%),32 (0.9%),344 (9.7%),<20,⋯,<20,2791 (78.4%),271 (7.6%),196.0,809 (22.7%),262 (7.4%),1274 (35.8%),546 (15.3%),0 (0%),0 (0%)
Asthma_51_64,Asthma,51_64,3430,58 (4.02),861 (25.1%),2527 (73.7%),42 (1.2%),30 (0.9%),760 (22.2%),<20,⋯,<20,2071 (60.4%),477 (13.9%),151.5,758 (22.1%),330 (9.6%),1613 (47%),593 (17.3%),0 (0%),0 (0%)
Asthma_18_35,Asthma,18_35,2072,30 (4.06),429 (20.7%),1542 (74.4%),101 (4.9%),40 (1.9%),491 (23.7%),<20,⋯,<20,1107 (53.4%),400 (19.3%),90.0,379 (18.3%),544 (26.3%),1360 (65.6%),128 (6.2%),0 (0%),0 (0%)
Asthma_36_50,Asthma,36_50,3254,43 (4.22),655 (20.1%),2517 (77.4%),82 (2.5%),60 (1.8%),717 (22%),<20,⋯,<20,1854 (57%),540 (16.6%),125.0,661 (20.3%),492 (15.1%),1688 (51.9%),445 (13.7%),0 (0%),0 (0%)
BreastC_65_105,BreastC,65_105,3238,73 (5.96),0 (0%),3172 (98%),66 (2%),33 (1%),319 (9.9%),<20,⋯,0 (0%),2576 (79.6%),192 (5.9%),188.0,625 (19.3%),182 (5.6%),1242 (38.4%),476 (14.7%),0 (0%),0 (0%)


# Richer SDoH survey Quintile demographics

In [28]:
# variables to reverse-code
to_flip <- c(
  "Health Coverage", "Health Literacy", "Healthcare Utilization",
  "Housing Quality", "Own Home", "Respect", "Social Cohesion",
  "Social Support", "Spiritual Experiences", "Walkability"
)

# -------------------------------------------------
# 1. Flip binary coding: 1 -> 0, 0 -> 1
# -------------------------------------------------
sdoh_survey_data_case_control[to_flip] <- lapply(
  sdoh_survey_data_case_control[to_flip],
  function(x) ifelse(is.na(x), NA, 1 - x)
)

# -------------------------------------------------
# 2. Rename variables after flipping
# -------------------------------------------------
new_names <- c(
  "No Health Coverage",
  "Low Health Literacy",
  "Low Healthcare Utilization",
  "Low Housing Quality",
  "No Own Home",
  "Low Respect",
  "Low Social Cohesion",
  "Low Social Support",
  "Low Spiritual Experiences",
  "Low Walkability"
)

sdoh_var <-c('No Own Home', 'Low Health Literacy', 'Low Housing Quality', 'Housing Instability',
             'Food Insecurity', 'Low Walkability', 'Loneliness', 'Crime', 'Physical Disorder', 'Social Disorder',
             'Everyday Discrimination', 'Medical Discrimination', 'Low Social Cohesion', 'Stress', 
             'Low Social Support', 'Low Spiritual Experiences', 'No Health Coverage', 'Low Healthcare Utilization',
             'Delayed Care', 'Can\'t afford care', 'Worried Pay', 'Low Respect'
)

names(sdoh_survey_data_case_control)[match(to_flip, names(sdoh_survey_data_case_control))] <- new_names

# -------------------------------------------------
# 3. Handle Binary Variables
# -------------------------------------------------

binary_vars <- c("Food Insecurity", "Can't afford care", "Housing Instability", "Low Health Literacy", 
                "Delayed Care", "Low Housing Quality", "No Health Coverage", "No Own Home")
       
sdoh_survey_data_case_control[, binary_vars] <- lapply(
  sdoh_survey_data_case_control[, binary_vars],
  function(x) as.numeric(
    ifelse(is.na(x), NA, ifelse(x < 0.5, 0, 1))
  )
)
              
# -------------------------------------------------
# 3. Z score continous variables
# -------------------------------------------------
       
continious_vars <- setdiff(sdoh_var, binary_vars)

sdoh_survey_data_case_control[continious_vars] <- lapply(
  sdoh_survey_data_case_control[continious_vars],
  function(x) as.numeric(scale(ifelse(is.na(x), NA, x)))
)
       
# -------------------------------------------------
# 4. Compute Q1 and Q5
# -------------------------------------------------

#Compute Q1 and Q5
quantiles <- sdoh_survey_data_case_control %>%
  pivot_longer(
    cols = all_of(sdoh_var),
    names_to = "domain",
    values_to = "value"
  ) %>%
  group_by(domain) %>%
  summarize(
    q1 = if (dplyr::n_distinct(stats::na.omit(value)) <= 2) {
      min(value, na.rm = TRUE)
    } else {
      round(as.numeric(stats::quantile(value, 0.20, na.rm = TRUE)), 3)
    },

    q5 = if (dplyr::n_distinct(stats::na.omit(value)) <= 2) {
      max(value, na.rm = TRUE)
    } else {
      round(as.numeric(stats::quantile(value, 0.80, na.rm = TRUE)), 3)
    },
    .groups = "drop"
  )



#Add Q1 / Q5 indicator columns back to original data
for (v in sdoh_var) {
  q1_val <- quantiles$q1[quantiles$domain == v]
  q5_val <- quantiles$q5[quantiles$domain == v]

  sdoh_survey_data_case_control[[paste0(v, "_Q1")]] <- sdoh_survey_data_case_control[[v]] <= q1_val
  sdoh_survey_data_case_control[[paste0(v, "_Q5")]] <- sdoh_survey_data_case_control[[v]] >= q5_val
}
       

In [29]:
table2 <- summarize_disease_sdoh_quintiles(
  cohort   = sdoh_survey_data_case_control,
  diseases = diseases_short,
  sdoh_vars = sdoh_var
)

table2 
    
#Save
write_excel_csv(
  table2,
  file.path(output_dir, "Case_richer_SDoH_quintiles_demographics.csv")
)

SDoH,Stratum,Disease,Sample size,"Age (median, sd)",Cisgender Male,Cisgender Female,SGM,Asian,Black,⋯,Native Hawaiian or Other Pacific Islander,White,Hispanic,Median Record Depth,Disability,LGBTQIA,Income <50k,Income >150k,Advanced Degree,Less than HighSchool
<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
No Own Home,Q1,Asthma,3735,63 (13.8),1030 (27.6%),2656 (71.1%),49 (1.3%),46 (1.2%),178 (4.8%),⋯,0 (0%),3219 (86.2%),187 (5%),160.0,806 (21.6%),354 (9.5%),694 (18.6%),953 (25.5%),0 (0%),0 (0%)
No Own Home,Q1,Afib,3205,73 (10.09),1889 (58.9%),1242 (38.8%),74 (2.3%),<20,103 (3.2%),⋯,<20,2933 (91.5%),59 (1.8%),173.0,950 (29.6%),205 (6.4%),701 (21.9%),657 (20.5%),0 (0%),0 (0%)
No Own Home,Q1,CKD,2796,73 (9.46),1397 (50%),1395 (49.9%),<20,26 (0.9%),266 (9.5%),⋯,<20,2333 (83.4%),76 (2.7%),205.0,906 (32.4%),130 (4.6%),809 (28.9%),428 (15.3%),0 (0%),0 (0%)
No Own Home,Q1,CHD,4837,73 (8.7),3081 (63.7%),1662 (34.4%),94 (1.9%),36 (0.7%),179 (3.7%),⋯,<20,4333 (89.6%),134 (2.8%),192.0,1476 (30.5%),320 (6.6%),1062 (22%),1018 (21%),0 (0%),0 (0%)
No Own Home,Q1,HyperC,10186,70 (10.31),4570 (44.9%),5473 (53.7%),143 (1.4%),116 (1.1%),494 (4.8%),⋯,<20,8923 (87.6%),324 (3.2%),172.0,2167 (21.3%),706 (6.9%),2101 (20.6%),2388 (23.4%),0 (0%),0 (0%)
No Own Home,Q1,t2d,5068,69 (10.63),2622 (51.7%),2373 (46.8%),73 (1.4%),78 (1.5%),461 (9.1%),⋯,<20,4076 (80.4%),298 (5.9%),177.0,1664 (32.8%),387 (7.6%),1395 (27.5%),757 (14.9%),0 (0%),0 (0%)
No Own Home,Q5,Asthma,1822,47 (15.51),354 (19.4%),1387 (76.1%),81 (4.4%),24 (1.3%),354 (19.4%),⋯,<20,1150 (63.1%),245 (13.4%),155.0,745 (40.9%),370 (20.3%),1142 (62.7%),108 (5.9%),0 (0%),0 (0%)
No Own Home,Q5,Afib,838,69 (14.18),394 (47%),419 (50%),25 (3%),<20,106 (12.6%),⋯,0 (0%),644 (76.8%),50 (6%),189.5,369 (44%),121 (14.4%),508 (60.6%),63 (7.5%),0 (0%),0 (0%)
No Own Home,Q5,CKD,1003,69 (12.42),432 (43.1%),567 (56.5%),<20,<20,275 (27.4%),⋯,0 (0%),612 (61%),74 (7.4%),224.0,488 (48.7%),88 (8.8%),711 (70.9%),37 (3.7%),0 (0%),0 (0%)
